# MissionCtrl — One-Click Training Notebook

**OpenEnv Hackathon Round 2**

This notebook trains an OverseerAgent to detect hallucinations in a multi-agent fleet using GRPO + Unsloth.

**Runtime required**: A100 GPU (HuggingFace compute credits)
**Expected training time**: ~2-3 hours
**Expected final reward**: 0.80+

**Recent optimizations**:
- ✅ Fixed FLAG_HALLUCINATION vs FLAG naming mismatch
- ✅ Fixed evaluate() reward accumulation bug
- ✅ Google Drive checkpoint persistence (saves every 50 steps)
- ✅ Platform-specific configs (Colab A100 vs Kaggle T4)
- ✅ Reduced curriculum for Kaggle 9-hour sessions

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Use unsloth[colab-new] for Colab-specific optimizations
!pip install "unsloth[colab-new]" trl openenv transformers datasets accelerate matplotlib --quiet
!pip install --upgrade bitsandbytes --quiet
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Clone / upload project files ─────────────────────────────────────
# Option A: Clone from your repo
# !git clone https://github.com/your-username/missionctrl .

# Option B: Upload environment.py, reward_model.py, train.py manually
# (Use the Files panel on the left in Colab)

# Verify files are present
import os
required = ['environment.py', 'reward_model.py', 'train.py']
for f in required:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status} {f}')

In [ ]:
# ── Cell 3: Verify GPU ────────────────────────────────────────────────────────
import torch
import os

# Set environment variable for tokenizer parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'GPU count: {torch.cuda.device_count()}')
assert torch.cuda.is_available(), 'No GPU detected — switch runtime to A100'
print('✅ GPU verified')

In [ ]:
# ── Cell 4: Smoke-test the environment ───────────────────────────────────────
from environment import MissionCtrlEnv, OverseerAction
from reward_model import compute_reward, reward_breakdown

env = MissionCtrlEnv(difficulty='medium', num_tasks=3, seed=42)
obs, info = env.reset()

print('Environment smoke test:')
print(f'  Tasks loaded: {len(obs["task_board"])}')
print(f'  Agent messages: {len(obs["recent_messages"])}')

# Test a FLAG action
first_task = obs['task_board'][0]['task_id']
action = OverseerAction('FLAG', task_id=first_task, evidence='fabricated citation detected')
obs2, reward, terminated, truncated, info = env.step(action)
print(f'  Step reward: {reward:.3f}')
print(f'  Info: {info}')
print('✅ Environment working correctly')

In [ ]:
# ── Cell 5: Run pre-training baseline ────────────────────────────────────────
from train import run_baseline

baseline_reward = run_baseline()
print(f'\n🎯 Baseline established: {baseline_reward:.3f}')
print('This is your starting floor. Training target: 0.75+')

In [ ]:
# ── Cell 6: Set HuggingFace credentials ──────────────────────────────────────
from huggingface_hub import login

# Option A: Use Colab secrets (recommended)
# 1. Go to the left panel → Secrets (🔑 icon)
# 2. Add a new secret named "HF_TOKEN" with your HuggingFace token
# 3. Run this cell:

import os
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"])
    print('✅ Logged in via HF_TOKEN secret')
else:
    # Option B: Interactive prompt
    login()
    print('✅ Logged in interactively')

# Set your repo name in train.py before running Cell 7
import train
train.HF_REPO = 'your-hf-username/missionctrl-overseer'  # ← CHANGE THIS
print(f'Will push to: {train.HF_REPO}')

In [ ]:
# ── Cell 7: TRAIN ─────────────────────────────────────────────────────────────
# Full 3-phase curriculum with reward-gated advancement.
# Watch the reward climb from ~0.31 → 0.80+
#
# Note: Checkpoints are automatically saved to Google Drive every 50 steps.
# If training is interrupted, you can resume from the last checkpoint.

from train import train

history = train()
print('\n🏆 Training complete!')

In [ ]:
# ── Cell 8: Display reward curve ─────────────────────────────────────────────
from IPython.display import Image

# Checkpoints are saved to Google Drive at:
# /content/drive/MyDrive/missionctrl_checkpoints/reward_curve.png
Image('/content/drive/MyDrive/missionctrl_checkpoints/reward_curve.png')

In [ ]:
# ── Cell 9: Before/After demo comparison ─────────────────────────────────────
# Load trained model and compare with baseline behavior
from unsloth import FastLanguageModel
from environment import MissionCtrlEnv, parse_action
from train import build_user_prompt, SYSTEM_PROMPT
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    './missionctrl_checkpoints/final',
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# Use a known hallucinated episode (seed 0, hard difficulty)
env = MissionCtrlEnv(difficulty='hard', num_tasks=4, seed=0)
obs, _ = env.reset()

prompt = tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_prompt(obs)},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3584).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)

completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('=== TRAINED MODEL OUTPUT ===')
print(completion)
print(f'\nParsed action: {parse_action(completion)}')

In [ ]:
# ── Cell 10: Final evaluation run ────────────────────────────────────────────
from train import evaluate

final_reward, metrics = evaluate(model, tokenizer, difficulty='hard', num_tasks=4, n_episodes=20)

print('\n=== FINAL EVALUATION SUMMARY ===')
print(f'  Overall reward:        {metrics["mean_reward"]:.3f} ± {metrics["std_reward"]:.3f}')
print(f'  Detection rate:        {metrics["mean_detect_rate"]:.1%}')
print(f'  False positive rate:   {metrics["mean_fp_rate"]:.1%}')